# 02 — Structures de données

Complexités des opérations Python, aliasing, copies, piles et files.

Chaque notion suit le schéma : **définition → quand l'utiliser → code testé (`assert`) → pièges → astuce**.


## 2.0 Tableau des coûts (liste Python)

| Opération | Complexité | Mécanisme |
|---|---|---|
| `L[i]`, `len(L)` | O(1) | accès direct par index |
| `L.append(x)`, `L.pop()` | O(1) amorti | tableau dynamique en fin |
| `L.insert(0, x)`, `L.pop(0)`, `del L[0]` | **O(n)** ⚠️ | décale les n éléments |
| `x in L`, `min(L)`, `max(L)`, `sum(L)` | O(n) | parcours séquentiel |
| `L[i:j]`, `L.copy()`, `list(L)` | O(j−i) / O(n) | copie physique |
| `L1 + L2` | O(n1 + n2) | nouvelle liste |
| `sorted(L)`, `L.sort()` | O(n log n) | Timsort |


## 2.1 Coût réel de `pop()` vs `pop(0)`

**Définition.** Retirer en fin de liste est O(1) ; retirer en tête décale tous les éléments, donc O(n).

**Quand l'utiliser.** `pop()` pour une pile ; jamais `pop(0)` dans une boucle (utiliser `deque`).


In [ ]:
import time

n = 20_000

# retrait en fin : O(1)
L = list(range(n))
t0 = time.perf_counter()
for _ in range(n // 10):
    L.pop()
t_fin = time.perf_counter() - t0
assert len(L) == n - n // 10

# retrait en tête : O(n) → beaucoup plus lent
L = list(range(n))
t0 = time.perf_counter()
for _ in range(n // 10):
    L.pop(0)
t_debut = time.perf_counter() - t0
assert len(L) == n - n // 10

print(f"pop()     en fin   : {t_fin:.4f} s")
print(f"pop(0)    en tête  : {t_debut:.4f} s")
print(f"ratio (théorique O(n)) : x{t_debut / max(t_fin, 1e-9):.1f}")


**Pièges.** `L.pop(0)` dans une boucle de n tours transforme un O(n) en **O(n²)**.

**Astuce.** Pour une file, `collections.deque` offre `popleft()` en O(1).


## 2.2 Concaténation : `+`, `+=`, `append`, `extend`, `join`

**Définition.** `+` crée une nouvelle liste ; `+=` modifie en place ; `append` ajoute un élément ; `extend` ajoute tous les éléments.

**Quand l'utiliser.** `append` pour un élément, `extend` pour une séquence, `"".join` pour construire une chaîne.


In [ ]:
L = [1, 2]
L.append([3, 4])
assert L == [1, 2, [3, 4]]      # un seul élément (une liste)

L = [1, 2]
L.extend([3, 4])
assert L == [1, 2, 3, 4]

mots = ["a", "b", "c"]
assert "-".join(mots) == "a-b-c"

# += modifie en place (même objet)
A = [1]
B = A
A += [2]
assert B == [1, 2]

# + crée un nouvel objet
A = [1]
B = A
A = A + [2]
assert B == [1]


**Pièges.** `s += ...` dans une boucle sur une chaîne crée une nouvelle chaîne à chaque tour : O(n²).

**Astuce.** Accumuler dans une liste puis `"".join(liste)` : coût total O(n).


## 2.3 Aliasing, copie superficielle et copie profonde

**Définition.** `B = A` crée un alias (même objet). `A.copy()` / `A[:]` crée une copie superficielle. `copy.deepcopy` copie récursivement.

**Quand l'utiliser.** Copier une liste avant de la modifier ; `deepcopy` pour des listes de listes.


In [ ]:
A = [1, 2, 3]
B = A              # aliasing
C = A.copy()       # copie superficielle
B[0] = 99
assert A[0] == 99 and C[0] == 1
assert A is B and A is not C

# piège classique : matrice mal construite
M_faux = [[0] * 3] * 3      # 3 fois la MÊME ligne
M_faux[0][0] = 1
assert M_faux[1][0] == 1    # toutes les lignes changent !

M_ok = [[0] * 3 for _ in range(3)]
M_ok[0][0] = 1
assert M_ok[1][0] == 0

# copie superficielle vs profonde
import copy
X = [[1, 2], [3, 4]]
Y = X.copy()
Z = copy.deepcopy(X)
X[0][0] = 100
assert Y[0][0] == 100       # sous-listes partagées
assert Z[0][0] == 1         # deepcopy indépendante


**Pièges.** `[[0] * n] * m` partage les lignes ; une copie superficielle partage les sous-objets.

**Astuce.** `A is B` teste l'identité (même objet), `A == B` teste l'égalité des valeurs.


## 2.4 Quel type choisir ?

| Type | Mutable | Ordonné | Usage |
|---|---|---|---|
| `list` | oui | oui | séquence modifiable |
| `tuple` | **non** | oui | données fixes, clé de dict, retour multiple |
| `str` | **non** | oui | texte |
| `dict` | oui | oui (insertion) | association clé → valeur, accès O(1) |
| `set` | oui | non | unicité, test `in` en O(1) |


In [ ]:
t = (1, 2)
try:
    t[0] = 5
except TypeError:
    pass
else:
    raise AssertionError("un tuple est immuable")

d = {"a": 1}
d["b"] = 2
assert d["a"] == 1 and d["b"] == 2

s = {1, 2, 2, 3}
assert s == {1, 2, 3}
assert 2 in s

# échange via tuple
a, b = 1, 2
a, b = b, a
assert (a, b) == (2, 1)


**Pièges.** Un `set` n'est pas ordonné ; un `tuple` ne peut pas être modifié ; une clé de `dict` doit être *hashable*.

**Astuce.** `set(L)` déduplique en O(n) ; `len(set(L)) != len(L)` détecte un doublon.


## 2.5 Pile (LIFO), file (FIFO) et parenthésage

**Définition.** Une pile retire le dernier entré (LIFO) ; une file retire le premier entré (FIFO).

**Quand l'utiliser.** Pile : appels récursifs, historique, parenthésage. File : parcours en largeur (BFS).


In [ ]:
from collections import deque

# pile : append / pop
pile = []
pile.append(10)
pile.append(20)
assert pile.pop() == 20          # dernier entré, premier sorti

# file : append / popleft
file = deque()
file.append(10)
file.append(20)
assert file.popleft() == 10      # premier entré, premier sorti

# application : vérifier un parenthésage
def bien_parenthese(s):
    pile = []
    paires = {")": "(", "]": "[", "}": "{"}
    for c in s:
        if c in "([{":
            pile.append(c)
        elif c in paires:
            if not pile or pile.pop() != paires[c]:
                return False
    return not pile

assert bien_parenthese("([]{})")
assert bien_parenthese("")
assert not bien_parenthese("([)]")
assert not bien_parenthese("((")
assert not bien_parenthese(")(")


**Pièges.** Une file implémentée avec `L.pop(0)` coûte O(n) par retrait ; oublier le test `not pile` provoque une erreur sur `")"`.

**Astuce.** `deque` offre `append` / `popleft` en O(1) aux deux extrémités.


## Récapitulatif

| Structure | Opération clé | Coût |
|---|---|---|
| `list` | accès `L[i]` | O(1) |
| `list` | `pop(0)` | O(n) ⚠️ |
| `deque` | `popleft()` | O(1) |
| `dict` | `d[clé]` | O(1) moyen |
| `set` | `x in s` | O(1) moyen |


## Version du module

Le même algorithme, importé depuis `src/revision/`, doit donner le même résultat. Lance `pip install -e .` avant d'exécuter cette cellule.

In [ ]:
# Version du module — même résultat que le code inline ci-dessus
from revision import File, Pile, bien_parenthese, compter_occurrences

assert bien_parenthese("([]{})")
assert not bien_parenthese("([)]")

pile = Pile()
pile.empiler(10)
pile.empiler(20)
assert pile.depiler() == 20

file = File()
file.enfiler(10)
file.enfiler(20)
assert file.defiler() == 10

assert compter_occurrences("abca") == {"a": 2, "b": 1, "c": 1}
print("revision.structures : OK")
